# Extraction des ventes de Seine-et-Marne (DVF 2025)

**Objectif :** extraire du fichier DVF 2025 (France entière) les ventes de la Seine-et-Marne, puis les enregistrer dans un fichier léger et rapide à relire pour la suite de l'analyse.

## Pourquoi la Seine-et-Marne ?

- **Pourquoi pas Noisiel seul ?** La commune compte trop peu de ventes par an pour obtenir des statistiques fiables et entraîner un modèle (mesure ci-dessous).
- **Pourquoi la Seine-et-Marne ?** Le département offre un volume de données confortable pour une première version, et une grande diversité de marchés, entre l'urbain de Marne-la-Vallée et les zones rurales.
- **Pourquoi un zoom sur Noisiel ?** C'est une commune que je connais : cette connaissance du terrain aide à repérer les incohérences et à interpréter les résultats.

## 1. Lecture du fichier par morceaux

Le fichier France entière compte plusieurs millions de lignes : le charger d'un coup risquerait de saturer la mémoire. Avec `chunksize`, pandas le lit **par morceaux de 100 000 lignes**. Chaque morceau est filtré sur le département 77, puis tous les morceaux filtrés sont assemblés avec `pd.concat()`.

Les colonnes d'**identifiants** (codes géographiques, anciens identifiants, numéros de lots et de volume, codes de culture spéciale) sont lues en texte avec `dtype`. Sans cela, chaque morceau devine ses types séparément, ce qui produit des `DtypeWarning` et des colonnes aux types mélangés.

In [1]:
import pandas as pd

In [2]:
morceaux = []

for chunk in pd.read_csv(
    "../data/raw/dvf-2025.csv.gz",
    chunksize=100_000,
    dtype={"code_departement": str, 
           "code_commune": str, 
           "code_postal": str, 
           "ancien_nom_commune" : str, 
           "ancien_id_parcelle" : str ,
           "ancien_code_commune" : str,
           "lot1_numero" : str, 
           "lot2_numero" : str, 
           "lot3_numero" : str, 
           "lot4_numero" : str, 
           "lot5_numero" : str,
           "numero_volume" : str, 
           "nature_culture_speciale" : str, 
           "code_nature_culture_speciale" : str },
):
    morceau_77 = chunk[chunk["code_departement"] == "77"]     
    morceaux.append(morceau_77)

df_77 = pd.concat(morceaux, ignore_index=True)

In [3]:
print(df_77.shape)
print(df_77["code_departement"].unique())

(64913, 40)
<ArrowStringArray>
['77']
Length: 1, dtype: str


Le résultat ne contient qu'un seul code département (`77`) : le filtre n'a laissé passer aucune autre ligne.

## 2. Volume de données

In [4]:
print(len(df_77)) 
print(df_77["id_mutation"].nunique())

64913
23529


In [5]:
df_noisiel = df_77[df_77["nom_commune"] == "Noisiel"]

print(len(df_noisiel)) 
print(df_noisiel["id_mutation"].nunique())

725
162


| Périmètre | Lignes | Ventes | Lignes par vente |
| --- | --- | --- | --- |
| Seine-et-Marne (77) | 64 913 | 23 529 | 2,8 |
| Noisiel | 725 | 162 | 4,5 |

- Noisiel ne représente que **0,7 %** des ventes du département, et ce sera encore moins après filtrage : le choix d'élargir au département est confirmé.
- *Question ouverte :* Noisiel compte 4,5 lignes par vente, contre 2,8 pour le département. Pourquoi ?

## 3. Sauvegarde en Parquet

Le résultat est enregistré au format **Parquet** dans `data/processed/`. Ce format est léger (2,4 Mo, contre 116 Mo pour le fichier France entière compressé), rapide à lire, et il **conserve les types des colonnes** : les codes restent du texte, sans avoir à repréciser `dtype`.

Les notebooks suivants partiront de ce fichier, sans relire le fichier brut. Il est ignoré par Git, car il est entièrement régénérable avec ce notebook.

In [6]:
df_77.to_parquet("../data/processed/dvf_77_2025.parquet", index = False )

In [7]:
df_verif = pd.read_parquet("../data/processed/dvf_77_2025.parquet")
print(df_verif.shape)
df_verif.dtypes

(64913, 40)


id_mutation                         str
date_mutation                       str
numero_disposition                int64
nature_mutation                     str
valeur_fonciere                 float64
adresse_numero                  float64
adresse_suffixe                     str
adresse_nom_voie                    str
adresse_code_voie                   str
code_postal                         str
code_commune                        str
nom_commune                         str
code_departement                    str
ancien_code_commune                 str
ancien_nom_commune                  str
id_parcelle                         str
ancien_id_parcelle                  str
numero_volume                       str
lot1_numero                         str
lot1_surface_carrez             float64
lot2_numero                         str
lot2_surface_carrez             float64
lot3_numero                         str
lot3_surface_carrez             float64
lot4_numero                         str


La relecture donne bien 64 913 lignes et 40 colonnes, et les codes sont toujours en texte.

## Conclusion

Ce notebook lit le fichier DVF 2025 France entière **par morceaux** (`chunksize`) pour ne pas saturer la mémoire, et en extrait les **64 913 lignes de Seine-et-Marne, soit 23 529 ventes**.

- **Volume :** Noisiel ne compte que 162 ventes (0,7 % du département), ce qui confirme qu'une étude limitée à la commune aurait été trop fragile.
- **Problème rencontré :** la lecture par morceaux produisait des colonnes aux types mélangés (`DtypeWarning`). Toutes les colonnes d'identifiants sont désormais lues en texte.
- **Résultat :** les données sont enregistrées dans `data/processed/dvf_77_2025.parquet`. Les notebooks suivants partiront de ce fichier, sans relire le fichier brut.

**Prochaine étape :** ces données sont encore brutes. Il reste à appliquer les règles de filtrage définies dans le notebook d'exploration (ventes d'un seul logement, ventes classiques, données complètes) avant l'analyse.